# ZeroID as ODIS Layers 1–2 — an executable walkthrough

This notebook demonstrates [ZeroID](https://github.com/highflame-ai/zeroid) using the
vocabulary of the [ODIS draft](https://github.com/cosai-oasis/ws4-odis/blob/main/RFCs/ODIS.md)
(Open Delegation & Identity Standard, CoSAI/OASIS WS4). Each section is an ODIS concept;
each code cell is the ZeroID call that implements it. The committed outputs are from a real
run against a local instance.

It is the executable companion to the
[role-capability statement](../../docs/odis/role-capability-statement.md), which maps every
ODIS requirement to code and tests — including the ones ZeroID does **not** meet.

**Prerequisites** (from the repo root):

```bash
make setup-keys          # ES256 + RSA signing keys into ./keys
docker compose up -d     # zeroid + postgres on localhost:8899
pip install requests pyjwt cryptography
```

> **Local dev trust model** — this compose deployment runs the admin plane (identities,
> agents, policies, signals — served at the server root, zeroid#318)
> *unauthenticated*: tenancy comes from client-supplied `X-Account-ID` / `X-Project-ID`
> headers, and the CIBA approval in §7 names its subject in the request body. That is a
> deliberate dev-mode convenience so one notebook can play both the agent and the
> administrator. In production the admin plane sits behind an authenticated session, and
> tenancy and approving subject are derived from verified claims — never from the caller's
> say-so. The committed outputs were generated against the stock compose configuration,
> including `token.require_dpop: false` (see §5).

| ODIS term | In this notebook |
|---|---|
| Agent Registration Record (§6.1) | a ZeroID identity + its `CredentialPolicy` |
| Agent Runtime Credential (§6.2) | a short-lived DPoP-bound access token |
| Delegation Record (§6.3) | an RFC 8693 exchange: `act` chain, `delegation_depth`, `mission_id` |
| Attestation gate (L1-03/11) | trust level raised by a verified attestation, required by policy |
| Revocation / kill switch (L1-12, L3-04/05) | a CAE signal cascading down the delegation tree |
| Audit lineage (CC-01/02) | the delegation explorer's per-JTI graph |

In [1]:
import base64, json, time, uuid, requests, jwt
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import serialization

BASE = "http://localhost:8899"
# ODIS trust_domain scoping: every control-plane call is tenant-scoped.
TENANT = {"X-Account-ID": "acct-demo", "X-Project-ID": "proj-demo", "X-User-ID": "demo-admin@example.com"}

def api(method, path, **kw):
    return requests.request(method, BASE + path, headers={**TENANT, **kw.pop("headers", {})}, **kw)

def b64u(b): return base64.urlsafe_b64encode(b).rstrip(b"=").decode()

def gen_key():
    """An agent's holder key (ODIS: proof-of-possession key). Private half never leaves the agent."""
    k = ec.generate_private_key(ec.SECP256R1())
    pem = k.public_key().public_bytes(serialization.Encoding.PEM,
        serialization.PublicFormat.SubjectPublicKeyInfo).decode()
    n = k.public_key().public_numbers()
    jwk = {"kty": "EC", "crv": "P-256", "x": b64u(n.x.to_bytes(32, "big")), "y": b64u(n.y.to_bytes(32, "big"))}
    return k, pem, jwk

# Display-only decode: no signature check. Real targets verify against the JWKS
# (the SDK companion's §3 shows that path) — never trust unverified claims.
def claims_of(tok): return jwt.decode(tok, options={"verify_signature": False})
def show(obj): print(json.dumps(obj, indent=2, default=str))

run = uuid.uuid4().hex[:6]  # unique names per run, so the notebook is re-runnable
print("zeroid:", requests.get(BASE + "/health").json()["status"], "| run id:", run)

zeroid: healthy | run id: e54982


## 1 · Agent Registration Record — Layer 1 · The Passport (ODIS §6.1)

ODIS's durable governance record for a logical agent (`agent_id`, lifecycle state, sponsor,
trust domain) is a ZeroID **identity**. Registration alone confers no authority: the agent is
born `unverified`, and its stable name is a SPIFFE/WIMSE URI
(`spiffe://{trust_domain}/{account}/{project}/{type}/{external_id}`) — ODIS's stable
`agent_id`, distinct from any runtime credential.

In [2]:
orch_key, orch_pem, orch_jwk = gen_key()
r = api("POST", "/agents/register", json={
    "name": "Orchestrator", "external_id": f"orch-{run}",
    "identity_type": "agent", "sub_type": "orchestrator",
    "allowed_scopes": ["data:read", "data:write"],
    "created_by": "demo-admin@example.com",     # ODIS L1-10: accountable sponsor
    "public_key_pem": orch_pem,                  # holder public key (L1-09)
})
orch = r.json()["identity"]; orch_id = orch["id"]; orch_apikey = r.json()["api_key"]
show({k: orch[k] for k in ("wimse_uri", "identity_type", "trust_level", "status", "owner_user_id")})

{
  "wimse_uri": "spiffe://highflame.ai/acct-demo/proj-demo/agent/orch-e54982",
  "identity_type": "agent",
  "trust_level": "unverified",
  "status": "active",
  "owner_user_id": "demo-admin@example.com"
}


## 2 · Registration-record governance fields — Layer 1 · The Passport (ODIS §6.1)

ODIS puts `permitted_delegation_modes`, lifetime bounds, and required assurance on the
registration record. ZeroID models these as a **CredentialPolicy** attached to the identity:
maximum TTL, permitted grant types, a scope ceiling, a **required trust level**, and a
**maximum delegation depth**. Note `client_credentials` stays in the allow-list — the
post-attestation bootstrap credential is itself policy-checked (everything fails closed,
including the server's own convenience issuance).

In [3]:
r = api("POST", "/credential-policies", json={
    "name": f"odis-demo-{run}",
    "max_ttl_seconds": 3600,                                   # L1-05: bounded lifetime
    "allowed_grant_types": ["api_key", "token_exchange", "client_credentials"],
    "allowed_scopes": ["data:read", "data:write"],              # scope ceiling
    "required_trust_level": "first_party",                     # L1-11: attestation-gated authority
    "max_delegation_depth": 2,                                  # Pillar 4: bounded cascade
})
policy = r.json()
api("PATCH", f"/identities/{orch_id}", json={"credential_policy_id": policy["id"]})
show({k: policy[k] for k in ("name", "max_ttl_seconds", "required_trust_level", "max_delegation_depth")})

{
  "name": "odis-demo-e54982",
  "max_ttl_seconds": 3600,
  "required_trust_level": "first_party",
  "max_delegation_depth": 2
}


## 3 · No registration resolution, no authority — The Bridge refuses (ODIS-L2-14 + L1-11)

ODIS: an Agent Runtime Credential is issued only after resolving to an **active** registration
that permits the request. The identity exists and is `active` — but its policy demands
`first_party` trust and the agent has never been attested. Issuance must refuse:

In [4]:
r = requests.post(BASE + "/oauth2/token", json={"grant_type": "api_key", "api_key": orch_apikey, "scope": "data:read"})
print(r.status_code); show(r.json())
assert r.status_code == 400

400
{
  "error": "policy_violation",
  "error_description": "credential policy violation: identity trust level \"unverified\" does not meet required level \"first_party\""
}


## 4 · Attestation raises trust — Layer 1 · The Passport (ODIS-L1-03 / L1-11)

ODIS Layer 1 gates credentials on attestation. ZeroID's production verifier is a generic,
fail-closed **OIDC workload attestation** (GitHub Actions, GCP Workload Identity Federation,
Kubernetes projected SA tokens — configured per tenant via `AttestationPolicy`; see
`docs/attestation.md`). A local notebook has no cloud control plane, so this run uses the
**dev-stub** proof type (`image_hash`) that ships for exactly this purpose — the flow,
promotion semantics, and policy gates are identical. Verification promotes trust
exactly-once inside a transaction: `hardware → first_party`.

In [5]:
att = api("POST", "/attestation/submit", json={
    "identity_id": orch_id, "level": "hardware",
    "proof_type": "image_hash", "proof_value": "sha256:deadbeef",   # dev stub: local demo only
}).json()
r = api("POST", "/attestation/verify", json={"attestation_id": att["id"]})
print("verify:", r.status_code)
print("trust level now:", api("GET", f"/identities/{orch_id}").json()["trust_level"])

verify: 200
trust level now: first_party


## 5 · Agent Runtime Credential — the Passport's output (ODIS §6.2, L1-05 / L1-09)

Short-lived, holder-bound, issued only post-attestation. The **DPoP proof** (RFC 9449) is
signed by the agent's holder key; the issued token carries `cnf.jkt` (the key thumbprint),
making it proof-of-possession rather than bearer. The `sub` is the WIMSE URI — logical agent
and runtime credential are distinct objects, which is ODIS's §1.3 identifier/credential
separation.

Deployment note: the Bearer *fallback* (omit the header, get an unbound token) can be
closed deployment-wide with `token.require_dpop: true` in zeroid.yaml — issuance without a
proof is then refused with `invalid_dpop_proof` and the AS metadata advertises
`dpop_bound_access_tokens_required` (RFC 9449 §5.1). This notebook's compose config keeps
the default (off) so the Bearer comparison in §6's delegation flow stays visible.

In [6]:
def dpop_proof(key, jwk, htm, htu):
    return jwt.encode({"jti": str(uuid.uuid4()), "htm": htm, "htu": htu, "iat": int(time.time())},
                      key, algorithm="ES256", headers={"typ": "dpop+jwt", "jwk": jwk})

proof = dpop_proof(orch_key, orch_jwk, "POST", f"{BASE}/oauth2/token")
r = requests.post(BASE + "/oauth2/token", headers={"DPoP": proof},
                  json={"grant_type": "api_key", "api_key": orch_apikey, "scope": "data:read data:write"})
tok = r.json(); orch_token = tok["access_token"]
c = claims_of(orch_token)
print("token_type:", tok["token_type"], "| expires_in:", tok["expires_in"], "s")
show({k: c[k] for k in ("sub", "trust_level", "mission_id", "jti")}); print("cnf:", c["cnf"])

token_type: DPoP | expires_in: 3600 s
{
  "sub": "spiffe://highflame.ai/acct-demo/proj-demo/agent/orch-e54982",
  "trust_level": "first_party",
  "mission_id": "2199e5ed-da96-4b07-a27a-9b124cbfa220",
  "jti": "2199e5ed-da96-4b07-a27a-9b124cbfa220"
}
cnf: {'jkt': 'RvPlc3-tDIahQ10Jk8rQt-G1NU6XkVXD7b8j4_Z_D8Q'}


A replayed proof must be rejected — the `jti` ledger is atomic (ODIS's replay concern
under L1-09; ZeroID's `TestDPoPReplayRejected` class of behavior, live):

In [7]:
r = requests.post(BASE + "/oauth2/token", headers={"DPoP": proof},
                  json={"grant_type": "api_key", "api_key": orch_apikey, "scope": "data:read"})
print(r.status_code); show(r.json())
assert r.status_code == 400

400
{
  "error": "invalid_dpop_proof",
  "error_description": "dpop: proof jti has already been observed within the freshness window (dpop_replay_detected)"
}


## 6 · Delegation Record — Layer 2 · The Bridge (ODIS §6.3, Pillar 4, L2-01/05/06)

The orchestrator delegates to a sub-agent via **RFC 8693 token exchange**. The sub-agent
proves possession of its own holder key (`actor_token`), and the granted scope is the
**three-way intersection**: requested ∩ what the orchestrator holds ∩ the sub-agent's policy
ceiling. The result carries ODIS Delegation-Record semantics as claims: `act` (the delegating
principal), `delegation_depth` (monotonic, capped by policy), and `mission_id` (chain lineage,
inherited from the root credential). Child expiry is clamped to the parent's — a child cannot
outlive its parent.

In [8]:
sub_key, sub_pem, _ = gen_key()
r = api("POST", "/agents/register", json={
    "name": "Researcher", "external_id": f"researcher-{run}", "identity_type": "agent",
    "sub_type": "tool_agent", "allowed_scopes": ["data:read"],
    "created_by": "demo-admin@example.com", "public_key_pem": sub_pem,
    "credential_policy_id": policy["id"],
})
researcher = r.json()["identity"]
att2 = api("POST", "/attestation/submit", json={"identity_id": researcher["id"], "level": "hardware",
    "proof_type": "image_hash", "proof_value": "sha256:cafef00d"}).json()
api("POST", "/attestation/verify", json={"attestation_id": att2["id"]})

actor_token = jwt.encode({"iss": researcher["wimse_uri"], "sub": researcher["wimse_uri"], "aud": BASE,
                          "iat": int(time.time()), "exp": int(time.time()) + 300}, sub_key, algorithm="ES256")
r = requests.post(BASE + "/oauth2/token", json={
    "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
    "subject_token": orch_token, "actor_token": actor_token, "scope": "data:read"})
delegated = r.json()["access_token"]; dc = claims_of(delegated)
show({k: dc[k] for k in ("sub", "act", "delegation_depth", "mission_id", "jti")})

{
  "sub": "spiffe://highflame.ai/acct-demo/proj-demo/agent/researcher-e54982",
  "act": {
    "sub": "spiffe://highflame.ai/acct-demo/proj-demo/agent/orch-e54982"
  },
  "delegation_depth": 1,
  "mission_id": "2199e5ed-da96-4b07-a27a-9b124cbfa220",
  "jti": "faa546a8-8ee0-4a77-a9ed-7ad801748d49"
}


Attenuation is monotonic and fails closed: the researcher's ceiling is `data:read`,
so a delegation that tries to smuggle `data:write` (which the *orchestrator* holds) is
refused — a sub-agent can never become a privilege-amplification point (ODIS-L2-06's core
demand):

In [9]:
r = requests.post(BASE + "/oauth2/token", json={
    "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
    "subject_token": orch_token, "actor_token": actor_token, "scope": "data:read data:write"})
print(r.status_code); show(r.json())
assert r.status_code == 400

400
{
  "error": "insufficient_scope",
  "error_description": "one or more requested scopes are not permitted for this identity: \"data:write\" not in allowed_scopes"
}


## 7 · Bounded asynchronous human authorization — Layer 2 · The Bridge (ODIS-L2-02, Pillar 1)

ODIS Pillar 1: agents need delegated human authority **without dragging the human into the
execution loop** — the Browser Trap. The Bridge's answer is CIBA (OpenID Client-Initiated
Backchannel Authentication): the agent requests authority out-of-band, a human approves
asynchronously (here via the admin surface; ping/push notifier modes exist), and only then
does polling yield a token. The `binding_message` shows the human exactly what they are
approving, and the issued token's `sub` is the **approving human** with the grant recorded
as `token_exchange: "ciba"` — originating-principal attribution, ODIS-CC-02.

> The `/approve` call naming its subject in the request body is the dev-mode admin surface
> (see the trust-model note at the top); a production approval UI derives the subject from
> the authenticated session. And if the post-approval poll returns `access_denied` on an
> *upgraded* checkout, your stored default credential policy predates the CIBA grant —
> `docker compose down -v` for a fresh volume (see the README).

In [10]:
alice_agent = api("POST", "/identities", json={
    "external_id": f"alice-proxy-{run}", "owner_user_id": "alice@example.com",
    "identity_type": "agent", "sub_type": "human_proxy",
    "allowed_scopes": ["data:read"], "trust_level": "first_party"}).json()
cl = api("POST", "/oauth/clients", json={
    "client_id": f"alice-proxy-{run}", "name": "Alice's proxy agent", "confidential": True,
    "identity_id": alice_agent["id"], "grant_types": ["urn:openid:params:grant-type:ciba"],
    "scopes": ["data:read"]}).json()

r = requests.post(BASE + "/oauth2/bc-authorize", json={
    "client_id": cl["client"]["client_id"], "client_secret": cl["client_secret"],
    "account_id": TENANT["X-Account-ID"], "project_id": TENANT["X-Project-ID"],
    "login_hint": "alice@example.com", "scope": "data:read",
    "binding_message": "Agent requests read access to the quarterly dataset"})
auth_req = r.json(); print("bc-authorize:", r.status_code); show(auth_req)

poll = lambda: requests.post(BASE + "/oauth2/token", json={
    "grant_type": "urn:openid:params:grant-type:ciba", "auth_req_id": auth_req["auth_req_id"],
    "client_id": cl["client"]["client_id"], "client_secret": cl["client_secret"]})
print("poll before approval:", poll().json()["error"])

api("POST", f"/oauth2/bc-authorize/{auth_req['auth_req_id']}/approve",
    json={"subject_id": "user-alice-001", "subject_email": "alice@example.com", "subject_name": "Alice"})
ciba_token = poll().json()
cc2 = claims_of(ciba_token["access_token"])
print("after approval — expires_in:", ciba_token["expires_in"], "s")
show({k: cc2.get(k) for k in ("sub", "token_exchange", "scope") if cc2.get(k)})

bc-authorize: 200
{
  "auth_req_id": "dUSvAUpFS9bZOBaUKdH4dgjua9TgW90mBfNSs-Nty8k",
  "expires_in": 300,
  "interval": 5
}


poll before approval: authorization_pending
after approval — expires_in: 900 s
{
  "sub": "user-alice-001",
  "token_exchange": "ciba"
}


## 8 · Confirmed compromise → cascade revocation — ODIS-L1-12, L3-04 / L3-05

ODIS: a confirmed compromise signal must revoke the affected credential and everything
derived from it. Ingesting a `critical` CAE signal against the **orchestrator** revokes its
credentials and cascades down the `parent_jti` tree — the researcher's *delegated* token dies
with its parent, and we time how long the whole thing takes:

In [11]:
t0 = time.time()
api("POST", "/signals/ingest", json={
    "identity_id": orch_id, "signal_type": "anomalous_behavior", "severity": "critical",
    "source": "notebook-demo", "payload": {"reason": "prompt injection detected"}})
child_state = requests.post(BASE + "/oauth2/token/introspect", json={"token": delegated}).json()
print(f"delegated token after cascade: {child_state}  ({(time.time()-t0)*1000:.0f} ms signal→dead)")
assert child_state == {"active": False}

delegated token after cascade: {'active': False}  (9 ms signal→dead)


## 9 · Audit lineage survives the kill — ODIS-CC-01 / CC-02

Revoked credentials are retained past expiry precisely so the delegation graph remains
walkable for forensics. The per-JTI lineage shows both identities (dual-identity audit),
the scope attenuation at each hop (`scopes_in` → `scopes_out`), and *why* each edge died:

In [12]:
g = api("GET", f"/delegations/by-jti/{dc['jti']}").json()
print("chain:", " → ".join(n["wimse_uri"].split("/")[-1] for n in g["nodes"]))
show([{k: e[k] for k in ("grant_type", "delegation_depth", "scopes_in", "scopes_out",
                          "attenuated", "is_revoked", "revoke_reason")} for e in g["edges"]])

chain: orch-e54982 → researcher-e54982
[
  {
    "grant_type": "api_key",
    "delegation_depth": 0,
    "scopes_in": [
      "data:read",
      "data:write"
    ],
    "scopes_out": [
      "data:read",
      "data:write"
    ],
    "attenuated": [],
    "is_revoked": true,
    "revoke_reason": "auto-revoked by CAE signal 5029913d-fd6b-46fa-aa48-57f4ee7c6393 (severity: critical)"
  },
  {
    "grant_type": "token_exchange",
    "delegation_depth": 1,
    "scopes_in": [
      "data:read",
      "data:write"
    ],
    "scopes_out": [
      "data:read"
    ],
    "attenuated": [
      "data:write"
    ],
    "is_revoked": true,
    "revoke_reason": "auto-revoked by CAE signal 5029913d-fd6b-46fa-aa48-57f4ee7c6393 (severity: critical)"
  }
]


## 10 · The sponsor leaves — offboarding kill switch (ODIS-L1-06 / L1-10 / L3-05)

ODIS-L1-10: every agent has an accountable human sponsor, and sponsor lifecycle events must
trigger de-provisioning. Alice — the human who just approved her proxy agent's authority in
§7 — leaves the organization. One call deactivates every identity she owned and
cascade-revokes their credentials, including the CIBA-approved token: the kill switch
(L3-05) keyed on the human, not the agent.

Reading the response correctly: `credentials_revoked` is **not** the cascade count. The
cascade runs *inside* each identity's deactivation; this counter only reports stragglers
caught by a final safety-net sweep, so `0` is the healthy value. The proof of revocation is
the credential itself — introspection reports it dead, and the audit graph (§9) records the
edge as revoked with the offboarding reason:

In [13]:
r = api("POST", "/identities/offboard-by-owner", json={"owner_user_id": "alice@example.com"})
show(r.json())
dead = requests.post(BASE + "/oauth2/token/introspect", json={"token": ciba_token["access_token"]}).json()
print("alice's CIBA-approved token:", dead)
assert dead == {"active": False}

g = api("GET", f"/delegations/by-jti/{cc2['jti']}").json()
show([{k: e[k] for k in ("grant_type", "is_revoked", "revoke_reason")} for e in g["edges"]])
assert all(e["is_revoked"] for e in g["edges"])

{
  "$schema": "http://localhost:8899/OffboardByOwnerOutputBody.json",
  "identities_deactivated": 1,
  "credentials_revoked": 0
}
alice's CIBA-approved token: {'active': False}
[
  {
    "grant_type": "urn:openid:params:grant-type:ciba",
    "is_revoked": true,
    "revoke_reason": "identity_deactivated"
  }
]


## What you just saw, in ODIS terms

| ODIS concept | Layer | Demonstrated |
|---|---|---|
| §6.1 Agent Registration Record; §1.3 identifier ≠ credential; CC-05 governed creation; L1-10 accountable sponsor | The Passport | §1 — registered identity, WIMSE `agent_id`, no authority conferred |
| §6.1 governance fields: lifetime bound, permitted grants, required trust, delegation-depth cap | The Passport | §2 — `CredentialPolicy` attached to the registration record |
| L2-14 registration resolution before authority; L1-11 fail closed | The Bridge | §3 — issuance refused for an unattested identity |
| L1-03 / L1-11 attestation-bootstrapped trust | The Passport | §4 — verification promoted trust, atomically |
| §6.2 Agent Runtime Credential; L1-05 / L1-09 proof-of-possession | The Passport | §5 — DPoP-bound token, replay rejected (`token.require_dpop` makes binding mandatory deployment-wide) |
| §6.3 Delegation Record; Pillar 4; L2-01 / L2-05 / L2-06 monotonic attenuation | The Bridge | §6 — `act` chain, depth, escalation refused |
| L2-02 bounded async authorization; Pillar 1 delegated principal identity | The Bridge | §7 — CIBA approval with binding message; token `sub` = the human |
| L1-12 / L3-04 / L3-05 compromise signal, cascade | (Router-adjacent) | §8 — critical signal killed the whole tree in milliseconds |
| CC-01 / CC-02 dual-identity audit lineage | cross-cutting | §9 — the graph outlives the credentials |
| L1-06 / L1-10 / L3-05 sponsor offboarding, kill switch | The Passport | §10 — the human leaves; every identity and credential they sponsored dies |

**What ZeroID deliberately does *not* demonstrate** — software/supply-chain attestation
(L1-02/08), bridge-mode provider adapters (L2-08..10), presenter isolation (L2-13), and
velocity limits (L3-03) are open gaps, documented with the same candor in the
[role-capability statement](../../docs/odis/role-capability-statement.md). That document is
the map; this notebook is the territory.